## Environments

In [ ]:
!pip install -q yolov5
!pip install tifffile

In [ ]:
!pip install pyyaml

# Import YAML module
import yaml

## Pre-processing Data

In [ ]:
import os
import shutil
import random

# Parameters
SOURCE_DIR = './standard_images/irradiation_20220509_A1/DUR_1_142'
TRAIN_DIR = './datasets/train'
VAL_DIR = './datasets/val'
VAL_RATIO = 0.2  # 20% for validation

# # Create train and val directories if they don't exist
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

# List all Grain_* folders
grain_folders = [f for f in os.listdir(SOURCE_DIR) if f.startswith('Grain_') and os.path.isdir(os.path.join(SOURCE_DIR, f))]

# Shuffle for randomness
random.shuffle(grain_folders)

# Split
val_count = int(len(grain_folders) * VAL_RATIO)
val_folders = grain_folders[:val_count]
train_folders = grain_folders[val_count:]

print(val_count)

# Move folders
for folder in train_folders:
    shutil.move(os.path.join(SOURCE_DIR, folder), os.path.join(TRAIN_DIR, folder))

for folder in val_folders:
    shutil.move(os.path.join(SOURCE_DIR, folder), os.path.join(VAL_DIR, folder))

print(f"Moved {len(train_folders)} folders to {TRAIN_DIR}/ and {len(val_folders)} folders to {VAL_DIR}/")


4
Moved 16 folders to ./datasets/train/ and 4 folders to ./datasets/val/


In [2]:
import os
import glob
import shutil
import pandas as pd
from PIL import Image, ImageSequence
import tifffile
import numpy as np

def process_split(grain_root, split):
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)

    grain_dirs = [os.path.join(grain_root, split, d) for d in os.listdir(os.path.join(grain_root, split))]
    
    for grain_dir in grain_dirs:
        # Find TIFF and CSV
        tiff_files = glob.glob(os.path.join(grain_dir, "*.tif*"))
        csv_files = glob.glob(os.path.join(grain_dir, "*.csv"))
        if not tiff_files or not csv_files:
            continue
        tiff_path = tiff_files[0]
        csv_path = csv_files[0]

        # Output image name
        img_basename = os.path.basename(grain_dir) + ".jpg"
        img_out_path = f"dataset/images/{split}/{img_basename}"

        # Load stack
        stack = tifffile.imread(tiff_path)

        def detect_white_layer(image_stack):
            white_threshold = 0.98
            max_intensity = image_stack.max()
            white_layers = []
            for i in range(image_stack.shape[0]):
                layer = image_stack[i]
                white_ratio = (layer > white_threshold * max_intensity).mean()
                if white_ratio > 0.9:
                    white_layers.append(i)
            return white_layers

        white_layers = detect_white_layer(stack)
        print(f"Detected white layers in {grain_dir}: {white_layers}")
        if len(white_layers) < 2:
            print(f"Skipping {grain_dir}: not enough white layers detected.")
            continue

        mica_slice = white_layers[1] + 1  # No +1 needed
        print(f"{grain_dir}: using slice {mica_slice}")

        # Use the mica slice from the stack
        image_array = stack[mica_slice]

        # Robust normalization using percentiles
        p_low, p_high = np.percentile(image_array, (1, 99))  # Ignore extreme outliers
        image_array = np.clip(image_array, p_low, p_high)    # Clip to this range
        image_array = (image_array - p_low) / (p_high - p_low) * 255
        image_array = image_array.astype(np.uint8)

        # Save as RGB
        image = Image.fromarray(image_array)
        image = image.convert("RGB")
        image.save(img_out_path)

        # Load CSV and convert to YOLO format
        df = pd.read_csv(csv_path)
        label_lines = []
        width, height = 1024, 752  # make sure these match the actual dimensions

        for _, row in df.iterrows():
            # Only process rows where Type is 'Mica'
            if row['Type'] != 'Mica':
                continue
            class_id = 0
            x, y = row['Y'], row['X']
            box_size = 32
            x_center = x / width
            y_center = y / height
            w_norm = box_size / width
            h_norm = box_size / height
            label_lines.append(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}")

        # Save labels
        label_out_path = f"dataset/labels/{split}/{os.path.basename(grain_dir)}.txt"
        with open(label_out_path, "w") as f:
            f.write("\n".join(label_lines))

# Run
process_split('datasets', 'train')
process_split('datasets', 'val')


Detected white layers in datasets\train\Grain_1: [27, 29]
datasets\train\Grain_1: using slice 30
Detected white layers in datasets\train\Grain_10: [43, 45]
datasets\train\Grain_10: using slice 46
Detected white layers in datasets\train\Grain_11: [32, 34]
datasets\train\Grain_11: using slice 35
Detected white layers in datasets\train\Grain_13: [38, 40]
datasets\train\Grain_13: using slice 41
Detected white layers in datasets\train\Grain_14: [31, 33]
datasets\train\Grain_14: using slice 34
Detected white layers in datasets\train\Grain_16: [32, 34]
datasets\train\Grain_16: using slice 35
Detected white layers in datasets\train\Grain_17: [32, 34]
datasets\train\Grain_17: using slice 35
Detected white layers in datasets\train\Grain_19: [22, 24]
datasets\train\Grain_19: using slice 25
Detected white layers in datasets\train\Grain_2: [32, 34]
datasets\train\Grain_2: using slice 35
Detected white layers in datasets\train\Grain_20: [27, 29]
datasets\train\Grain_20: using slice 30
Detected white

## Training

In [ ]:
%cd yolov5
%python train.py --img 640 --batch 16 --epochs 100 --data ../dataset.yaml --weights yolov5s.pt